# Texas Hold'em: game report

This notebook **is** the report. Point it at a finished run and choose **Run All**. Every table and chart then appears below as a cell output. It writes no files and does not start or change any game.

**How to use**

This notebook is a **report template**. The generic copy lives in `report-templates/<definition-id>/<template-id>/`. A report for one run is a copy of it inside that run's folder. The report workflow makes that copy and sets `RUN_DIR` for you. To run a copy by hand:

1. In the configuration cell below, set `RUN_DIR` to a finished instance directory, for example `~/.prime-env-harness/environments/instances/<instance-id>`.
2. Run all cells (Jupyter: *Run → Run All Cells*; or `jupyter nbconvert --to notebook --execute --inplace report.ipynb`).

The notebook accepts only finished runs (protocol 2; config `hands`, `starting_chips`, `small_blind`, `big_blind`; 2–9 seats). It stops with an error on anything else, including a game still in progress, so live private cards are never shown. Every fact comes from replaying the run's receipts through **the environment's own engine** (`environment.py` and `poker_eval.py` from the run's definition version). The replay must reproduce every recorded result, notice and the final snapshot exactly.

**Contents**

| § | Section |
|---|---|
| 0 | Configuration, loading, replay and verification |
| 1 | Run settings and seats |
| 2 | Final result and chip stacks |
| 3 | Hand by hand |
| 4 | Where the chips came from: position, showdown vs uncontested, biggest pots |
| 5 | Playing style |
| 6 | Decision quality: own-information equity vs price |
| 7 | Showdowns: equity by street against the hands that were revealed |
| 8 | Time to decide |
| 9 | Cost and token usage |
| 10 | Action mix and rejections |
| 11 | Complete action log |
| 12 | Verification results |
| 13 | Methods and limitations |

## 0. Configuration

This is the only cell you need to edit.

In [ ]:
RUN_DIR = ""  # report-parameters

### 0.1 Setup

This cell holds the imports, the chart style and the display helpers. Seats take the categorical colours in roster order. Gray is reserved for reference lines and for a ninth seat, which also gets a dashed line.

In [ ]:
import html
import importlib
import itertools
import json
import math
import random
import statistics
import sys
import warnings
from bisect import bisect_left
from collections import Counter, defaultdict
from copy import deepcopy
from datetime import datetime, timezone
from pathlib import Path

warnings.filterwarnings("ignore", message=".*numexpr.*")
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import HTML, Markdown, display
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm
from matplotlib.ticker import MaxNLocator

RUN_DIR = Path(RUN_DIR).expanduser().resolve()

SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
NEUTRAL, GRID, INK2 = "#9a9893", "#e6e5e1", "#52514e"
DIVERGING = LinearSegmentedColormap.from_list("chips", ["#e34948", "#f0efec", "#2a78d6"])
plt.rcParams.update({
    "figure.dpi": 110, "figure.facecolor": "white", "axes.facecolor": "white",
    "font.size": 9.5, "axes.titlesize": 10.5, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "axes.edgecolor": "#c9c8c3", "axes.linewidth": 0.8, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6, "axes.axisbelow": True,
    "axes.labelcolor": INK2, "xtick.color": INK2, "ytick.color": INK2,
    "legend.frameon": False, "lines.linewidth": 2,
})
STREETS = ["preflop", "flop", "turn", "river"]
MOVES = ["fold", "check", "call", "bet/raise"]
SUIT = {"c": "♣", "d": "♦", "h": "♥", "s": "♠"}

display(HTML("""<style>
.card{font-family:ui-monospace,Menlo,monospace;padding:0 3px;margin:0 1px;border:1px solid #c9c8c3;
      border-radius:3px;background:#fff;color:#0b0b0b;white-space:nowrap}
.card.red{color:#c4302b}.card.hidden{color:#9a9893;border-style:dashed}.private{opacity:.6}
.dataframe td,.dataframe th{padding:3px 8px;vertical-align:top}
.txt{text-align:left}.cell-left{text-align:left;white-space:nowrap}.cell-wrap{text-align:left;min-width:260px;max-width:420px}
</style>"""))


def num(x, places=2):
    """Display-only rounding with a Unicode minus."""
    if x is None or (isinstance(x, float) and math.isnan(x)):
        return "–"
    text = str(x) if isinstance(x, int) else f"{x:.{places}f}"
    return text.replace("-", "−")


def snum(x, places=2):
    """Signed display: +12, −3.50, 0."""
    text = num(x, places)
    if text.startswith("−") or text == "–" or not text.strip("0."):
        return text
    return "+" + text


def pct(part, whole):
    return 100 * part / whole if whole else None


def show(df, places=2, signed=(), escape=True, collapsed=None):
    """Render a DataFrame as a static HTML table (persists in the saved notebook and the HTML export)."""
    out = df.copy()
    for col in out.columns:
        fmt = snum if col in signed else num

        def cell(v, fmt=fmt):
            if v is None or (isinstance(v, (int, float)) and not isinstance(v, bool)):
                return fmt(v, places)
            return f"<div class='txt'>{html.escape(str(v)) if escape else v}</div>"
        out[col] = out[col].map(cell)
    table = out.to_html(index=False, escape=False, classes="dataframe", border=0)
    if collapsed:
        table = f"<details><summary>{collapsed}</summary>{table}</details>"
    display(HTML(table))


def finish(fig):
    fig.tight_layout()
    display(fig)
    plt.close(fig)


def utc(ms):
    return datetime.fromtimestamp(ms / 1000, tz=timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")


def ts_ms(iso):
    return round(datetime.fromisoformat(iso.replace("Z", "+00:00")).timestamp() * 1000)


def card_html(c, hidden=False):
    if hidden:
        return "<span class='card hidden'>??</span>"
    return f"<span class='card{' red' if c[1] in 'dh' else ''}'>{c[0].replace('T', '10')}{SUIT[c[1]]}</span>"


def cards_html(cs, hidden=False):
    if hidden:
        return card_html(None, True) * 2
    return "".join(card_html(c) for c in cs) if cs else "—"


def esc(x):
    return html.escape(str(x))


print("RUN_DIR", RUN_DIR)

### 0.1b Template guard

This check applies when the notebook sits next to a `template.json`, as it does in an instance's `reports/` folder. The run must then be an instance of that template's definition, on a version the template lists. A trial copy (`"trial": true`) may run on an unlisted version, so that the template can be verified against it.

In [ ]:
_tpl_path = Path("template.json")
if not (RUN_DIR / "instance.json").exists():
    raise SystemExit(f"set RUN_DIR to a finished instance directory (no instance.json under {RUN_DIR})")
if _tpl_path.exists():
    _tpl, _inst = json.loads(_tpl_path.read_text()), json.loads((RUN_DIR / "instance.json").read_text())
    if _inst.get("definition_id") != _tpl["definition_id"]:
        raise SystemExit(f"this template is for {_tpl['definition_id']!r}, not {_inst.get('definition_id')!r}")
    if _inst.get("version") not in _tpl["compatible_versions"] and not _tpl.get("trial"):
        raise SystemExit(f"template {_tpl['id']!r} is not verified for version {str(_inst.get('version'))[:12]}")
    print(f"template {_tpl['id']} (revision {_tpl['revision']}) on {_inst.get('definition_id')} "
          f"version {str(_inst.get('version'))[:12]}" + (" [trial]" if _tpl.get("trial") else ""))
else:
    print("no template.json beside this notebook: running standalone, version not checked")

### 0.2 Load and validate the run, and load the environment's own engine

This cell rejects any run that is unfinished or does not match the expected configuration. It then imports the engine (`environment.py`) and the hand evaluator (`poker_eval.py`) from the definition version the run used: `definitions/<definition-id>/<version>/source`. All the private-card displays come after this check.

In [ ]:
def _require(path, what):
    if not path.exists():
        raise SystemExit(f"missing {what}: {path}")
    return path


def _fail(msg):
    raise SystemExit(f"cannot report this run: {msg}")


checkpoint = json.loads(_require(RUN_DIR / "checkpoint.json", "checkpoint.json").read_text())
instance = json.loads(_require(RUN_DIR / "instance.json", "instance.json").read_text())
run_status = json.loads((RUN_DIR / "run.json").read_text()) if (RUN_DIR / "run.json").exists() else {}
snapshot, receipts = checkpoint["snapshot"], checkpoint["receipts"]

if checkpoint.get("protocol_version") != 2:
    _fail(f"unsupported protocol_version {checkpoint.get('protocol_version')!r}")
if not checkpoint.get("finished"):
    _fail("run is not finished (checkpoint['finished'] is empty)")
if snapshot.get("phase") != "finished":
    _fail(f"snapshot phase is {snapshot.get('phase')!r}, expected 'finished'")
cfg = snapshot.get("config") or {}
if set(cfg) != {"hands", "starting_chips", "small_blind", "big_blind"}:
    _fail(f"unexpected config fields: {sorted(cfg)}")
if any(type(v) is not int or v < 1 for v in cfg.values()):
    _fail(f"config values must be positive integers: {cfg}")
players = list(snapshot.get("seats") or [])
if not 2 <= len(players) <= 9 or len(set(players)) != len(players) or set(snapshot.get("players", {})) != set(players):
    _fail(f"expected 2–9 distinct seats, got {players}")
results = snapshot.get("results") or []
if len(results) != snapshot.get("hand_number") or not results:
    _fail(f"{len(results)} settled hands but hand_number is {snapshot.get('hand_number')}")

SOURCE = _require(RUN_DIR.parent.parent / "definitions" / instance["definition_id"] / instance["version"] / "source",
                  "definition source for this run's version")
for _name in ("environment", "sdk", "poker_eval"):
    sys.modules.pop(_name, None)
sys.path.insert(0, str(SOURCE))
try:
    env_mod = importlib.import_module("environment")
    sdk = importlib.import_module("sdk")
    poker_eval = importlib.import_module("poker_eval")
finally:
    sys.path.remove(str(SOURCE))
if not hasattr(env_mod, "Holdem"):
    _fail("the definition's environment.py has no Holdem engine")
rank_hand, hand_name = poker_eval.rank_hand, poker_eval.hand_name
ACTIONS = [a["name"] for a in env_mod.Holdem().describe_actions()]

seat_meta = {s["id"]: s for s in (instance.get("roster") or {}).get("seats", [])}
N_HANDS, BB = len(results), cfg["big_blind"]
CHIPS_TOTAL = len(players) * cfg["starting_chips"]
SEAT_COLOR = {p: SERIES[i] if i < len(SERIES) else NEUTRAL for i, p in enumerate(players)}
SEAT_STYLE = {p: "-" if i < len(SERIES) else "--" for i, p in enumerate(players)}
print(f"finished run: {len(players)} seats {players}, {N_HANDS}/{cfg['hands']} hands, {len(receipts)} receipts")
print("config:", cfg)
print("engine:", SOURCE, "actions", ACTIONS)

### 0.3 Read the seat transcripts and recover each seat's own hole cards

Earlier hands' undealt decks are not kept in the final snapshot, so hole cards come from three sources only:

* the seat's own wake views and tool responses (`hole_cards`), which show what the seat saw while it played;
* showdown reveals;
* the final snapshot, which still holds the last hand's cards.

A seat that folded without ever being woken in a hand (for example, a big blind that everyone folded to) has no recorded holding. That holding is shown as `??` and left out of every equity figure. Only state fields are read from the transcripts. Assistant text and reasoning are never read into the report.

In [ ]:
transcript_records = {}
for p in players:
    path = RUN_DIR / "seats" / p / "conversation.jsonl"
    transcript_records[p] = ([json.loads(line) for line in path.read_text().splitlines() if line.strip()]
                             if path.exists() else [])
print("seat transcripts:", {p: len(r) for p, r in transcript_records.items()}, "records")


def embedded_state(text):
    """The seat view carried by a wake message or a tool response, or None."""
    if text.startswith("Environment wake:\n"):
        text = text.split("\n", 1)[1]
    elif "Current state: " in text:
        text = text.split("Current state: ", 1)[1]
    try:
        data = json.JSONDecoder().raw_decode(text.lstrip())[0]
    except ValueError:
        return None
    state_ = data.get("state", data) if isinstance(data, dict) else None
    return state_ if isinstance(state_, dict) and "hole_cards" in state_ else None


seen_holes, hole_source, hole_conflicts = defaultdict(dict), defaultdict(dict), []


def note_hole(hand, seat, cards, source):
    if not cards or not hand:
        return
    previous = seen_holes[hand].get(seat)
    if previous is not None and previous != list(cards):
        hole_conflicts.append((hand, seat, previous, cards, source))
    seen_holes[hand][seat] = list(cards)
    hole_source[hand].setdefault(seat, source)


wake_records = []   # (seat, table seq the wake reflects, state, record id)
for p, recs in transcript_records.items():
    for rec in recs:
        msg = rec.get("message") or {}
        if rec.get("customType") == "environment_wake":
            st = embedded_state(rec.get("content", ""))
            if st is not None:
                wake_records.append((p, (rec.get("details") or {}).get("through_seq"), st, rec.get("id"),
                                    ts_ms(rec.get("timestamp"))))
        elif msg.get("role") == "toolResult" and msg.get("content"):
            st = embedded_state(msg["content"][0].get("text", ""))
        else:
            continue
        if st is not None:
            note_hole(st.get("hand_number"), p, st.get("hole_cards"), "own view")
for r in results:
    for s, shown in r["showdown"].items():
        note_hole(r["hand_number"], s, shown["cards"], "showdown")
for s, pl in snapshot["players"].items():
    note_hole(snapshot["hand_number"], s, pl["hole"], "final snapshot")
if hole_conflicts:
    _fail(f"inconsistent hole cards across sources: {hole_conflicts[:2]}")

# Public table log: commit time of each environment operation, and which operation each wake reflects.
# A harness status event (such as an idle-turn retry) has no operation; it reflects the latest one before it.
commit_ms, table_op, status_wakes, last_op = {}, {}, Counter(), None
TABLE_PATH = RUN_DIR / "table.jsonl"
if TABLE_PATH.exists():
    for line in TABLE_PATH.read_text().splitlines():
        if line.strip():
            ev = json.loads(line)
            if ev.get("op_seq") is not None:
                last_op = ev["op_seq"]
                commit_ms[last_op] = min(ev["timestamp"], commit_ms.get(last_op, ev["timestamp"]))
            elif ev.get("wake") and isinstance(ev.get("to"), list):
                for seat in ev["to"]:
                    status_wakes[seat] += 1
            table_op[ev["seq"]] = last_op
print(f"hole cards recovered for {sum(len(v) for v in seen_holes.values())} seat-hands; "
      f"{len(wake_records)} wake states to verify")

### 0.4 Replay every receipt through the environment's own engine

`ReplayHoldem` is the run's own `Holdem` class. It changes one thing: each hand's dealt cards are set from the run's records instead of a fresh shuffle. The recorded board and the recovered hole cards are dealt, and unused cards stand in for burns and for holdings that were never recorded. All betting, pot, side-pot, refund and settlement logic is the environment's own.

Receipts are replayed in sequence order. For each one the replay checks that:

* accepted actions return the recorded result, notices and progress;
* rejected actions raise the recorded error and leave the state unchanged;
* chips are conserved after every operation.

The rebuilt final state must equal `checkpoint.snapshot` exactly. The only exception is the undealt remainder of the last deck, which was never recorded.

In [ ]:
FULL_DECK = [r + s for r in "23456789TJQKA" for s in "cdhs"]
hand_meta = {}


class ReplayHoldem(env_mod.Holdem):
    """The environment's own engine; only the dealt cards come from the run's records."""

    def _start_hand(self):
        super()._start_hand()
        st = self.state
        if st["phase"] == "finished":
            return
        h = st["hand_number"]
        board, known = results[h - 1]["board"], seen_holes.get(h, {})
        dealt = [s for s in st["seats"] if st["players"][s]["hole"]]
        used = set(board) | {c for s in dealt for c in known.get(s, [])}
        spare = [c for c in FULL_DECK if c not in used]
        for s in dealt:
            st["players"][s]["hole"] = list(known[s]) if s in known else [spare.pop(), spare.pop()]
        runout = board + [spare.pop() for _ in range(5 - len(board))]
        burns = [spare.pop() for _ in range(3)]
        # pop() deals from the end: burn, flop x3, burn, turn, burn, river.
        st["deck"] = spare + [runout[4], burns[2], runout[3], burns[1], runout[2], runout[1], runout[0], burns[0]]
        hand_meta[h] = {"button": st["seats"][st["button"]], "sb": st["small_blind_seat"],
                        "bb": st["big_blind_seat"], "dealt": dealt, "hidden": [s for s in dealt if s not in known],
                        "start_stacks": dict(st["hand_start_stacks"]),
                        "holes": {s: list(st["players"][s]["hole"]) for s in dealt}}


def position_of(seat, hand):
    m = hand_meta[hand]
    if seat == m["button"]:
        return "Button (SB)" if seat == m["sb"] else "Button"
    return {m["sb"]: "Small blind", m["bb"]: "Big blind"}.get(seat, "Other")


def move_of(action, street_total, bet_before):
    if action in ("fold", "check"):
        return action
    return "bet/raise" if action in ("raise_to", "all_in") and street_total > bet_before else "call"


wakes_by_op = defaultdict(list)
for p, through, st, rid, ts in wake_records:
    wakes_by_op[table_op.get(through)].append((p, st, rid, ts))

game = ReplayHoldem()
attempts, settled_at, wake_problems, wakecount = [], {}, [], 0
late_wakes, actor_at, pending_late = [], {}, []
views_after_op = {}
n_receipts = Counter()


def check(ok, what):
    if not ok:
        raise AssertionError(what)


try:
    for rid, r in sorted(receipts.items(), key=lambda x: x[1]["sequence"]):
        req, method, seq = r["request"], r["request"]["method"], r["sequence"]
        n_receipts[method] += 1
        p = req.get("participant")
        before = deepcopy(game.__dict__.get("state"))
        n_results = len(before["results"]) if before else 0
        row = None
        sdk._begin_request(method)
        try:
            if method == "initialize":
                result = game.initialize(req["params"])
            elif method == "invoke":
                st = game.state
                pl = st["players"][p]
                a, args = req["params"]["action"], req["params"]["arguments"]
                h = st["hand_number"]
                row = dict(sequence=seq, receipt_id=rid, hand=h, street=st["phase"], seat=p, action=a,
                           amount=args.get("amount") if isinstance(args, dict) else None,
                           pot=sum(q["total_paid"] for q in st["players"].values()),
                           to_call=min(max(0, st["current_bet"] - pl["street_paid"]), pl["stack"]),
                           stack=pl["stack"], current_bet=st["current_bet"],
                           n_opponents=sum(1 for s in game._live() if s != p),
                           hole=list(pl["hole"]), hole_known=p in seen_holes.get(h, {}), board=list(st["board"]),
                           legal=game._legal(p), position=position_of(p, h) if h in hand_meta else "–")
                result = game.invoke(p, a, args)
            elif method == "handle_input":
                result = game.handle_input(p, req["params"]["text"])
            elif method == "observe":
                result = game.observe(p)
            else:
                _fail(f"unexpected receipt method {method!r}")
            ok = True
        except ValueError as exc:
            result, ok = str(exc), False
        check(ok == r["success"], f"seq {seq}: success {ok} but recorded {r['success']}")
        if ok:
            check(result == r["result"], f"seq {seq}: result differs: {result} vs {r['result']}")
            recorded = {k: r[k] for k in ("notices", "progress", "finished") if k in r}
            check(sdk._response_metadata() == recorded, f"seq {seq}: notices/progress differ")
        else:
            check(result == r.get("error", r.get("result")), f"seq {seq}: error differs: {result}")
            check(game.state == before, f"seq {seq}: a rejected attempt changed state")
        if game.__dict__.get("state"):
            check(sum(q["stack"] + q["total_paid"] for q in game.state["players"].values()) == CHIPS_TOTAL,
                  f"seq {seq}: chips not conserved")
            if len(game.state["results"]) > n_results:
                settled_at[len(game.state["results"])] = seq
        if row is not None:
            row.update(success=ok, result=result)
            if ok:
                ev = result["action"]
                row.update(paid=ev["paid"], street_total=ev["street_total"],
                           move=move_of(a, ev["street_total"], row["current_bet"]))
            attempts.append(row)
        if ok and row is not None:
            actor_at[seq] = p
        # Every seat's view after this operation, kept so a wake that was queued while its
        # seat was still acting can be matched against the view it was delivered with.
        views_after_op[seq] = {q: game.agent_view(q) for q in players}
        for seat, wake_state, wid, wake_ts in wakes_by_op.get(seq, []):
            if game.agent_view(seat) == wake_state:
                wakecount += 1
            else:
                pending_late.append({"seat": seat, "id": wid, "seq": seq, "state": wake_state, "ts": wake_ts})
    final = game.snapshot()
    EXCLUDED = ("deck", "decks")
    diff = [k for k in set(final) | set(snapshot) if k not in EXCLUDED and final.get(k) != snapshot.get(k)]
    check(not diff, f"final state differs from checkpoint.snapshot in {diff}")
    check(all(len(final[k]) == len(snapshot[k]) for k in EXCLUDED), "undealt card counts differ")
except AssertionError as exc:
    raise SystemExit("cannot report this run: its receipts do not replay through this version's engine; "
                     f"first failed check: {str(exc)[:240]}") from None
# A wake queued while its seat was still acting is handed over only later, carrying the view
# of the operation that had committed at that moment; if the seat acted again first, that is
# its own next operation. Verify such wakes against that view and note how far they lag.
_wake_ops = sorted(commit_ms)


def _latest_op_at(when):
    best = None
    for op in _wake_ops:
        if commit_ms[op] <= when:
            best = op
        else:
            break
    return best


for pending in pending_late:
    seat, op, state = pending["seat"], pending["seq"], pending["state"]
    delivered_at = _latest_op_at(pending["ts"]) if pending["ts"] is not None else None
    if views_after_op.get(op + 1, {}).get(seat) == state and actor_at.get(op + 1) == seat:
        late_wakes.append((seat, pending["id"], op, op + 1))
    elif delivered_at is not None and delivered_at >= op and views_after_op.get(delivered_at, {}).get(seat) == state:
        late_wakes.append((seat, pending["id"], op, delivered_at))
    else:
        wake_problems.append((seat, pending["id"], op))
if wake_problems:
    raise SystemExit(f"wake verification failed for {len(wake_problems)} records: {wake_problems[:3]}")
wakecount += len(late_wakes)
unmatched_wakes = sum(len(v) for k, v in wakes_by_op.items() if k not in {r["sequence"] for r in receipts.values()})

decisions = [a for a in attempts if a["success"]]
print(f"replay OK: {dict(n_receipts)} receipts, {len(decisions)} accepted actions, "
      f"{len(attempts) - len(decisions)} rejected; final stacks {{{', '.join(f'{s}: {q['stack']}' for s, q in final['players'].items())}}}")
_lags = [delivered - op for _, _, op, delivered in late_wakes]
print(f"verified {wakecount} wake states against the replayed agent view"
      + (f", {len(late_wakes)} of them late queued wakes carrying the view of the operation committed when they "
         f"were delivered (up to {max(_lags)} operation(s) past their own)" if late_wakes else "")
      + f" ({unmatched_wakes} wakes had no operation to compare)")

### 0.5 Equity estimates

**Own-information equity** is the share of the pot a seat would win against the opponents still in the hand, if each held a uniformly random holding. It uses only what the seat could see: its own hole cards and the board. It is estimated by Monte Carlo with a fixed seed per decision, so reruns give the same numbers. The sample count shrinks for very long runs so that the cell stays fast.

**Showdown equity** uses the hands revealed at showdown, so it is a hindsight measure. It is enumerated exactly on the flop and turn and sampled preflop.

In [ ]:
def equity_vs_random(hole, board, n_opp, samples, seed):
    rng = random.Random(seed)
    unseen = [c for c in FULL_DECK if c not in hole and c not in board]
    need = 5 - len(board)
    won = 0.0
    for _ in range(samples):
        draw = rng.sample(unseen, need + 2 * n_opp)
        runout = board + draw[:need]
        mine = rank_hand(hole + runout)
        theirs = [rank_hand(draw[need + 2 * i: need + 2 * i + 2] + runout) for i in range(n_opp)]
        best = max(theirs)
        if mine > best:
            won += 1
        elif mine == best:
            won += 1 / (1 + theirs.count(mine))
    return won / samples


def equity_known(holes, board, samples, seed):
    """Exact (≤ 2 cards to come) or sampled pot share for each revealed hand."""
    used = set(board) | {c for cs in holes.values() for c in cs}
    unseen = [c for c in FULL_DECK if c not in used]
    need = 5 - len(board)
    if need <= 2:
        runouts = list(itertools.combinations(unseen, need))
    else:
        rng = random.Random(seed)
        runouts = [rng.sample(unseen, need) for _ in range(samples)]
    share = dict.fromkeys(holes, 0.0)
    for extra in runouts:
        full = board + list(extra)
        ranks = {s: rank_hand(cs + full) for s, cs in holes.items()}
        top = max(ranks.values())
        winners = [s for s, v in ranks.items() if v == top]
        for s in winners:
            share[s] += 1 / len(winners)
    return {s: v / len(runouts) for s, v in share.items()}


scorable = [d for d in decisions if d["hole_known"] and d["n_opponents"] > 0]
EQ_SAMPLES = int(min(2000, max(300, 400_000 // max(1, len(scorable)))))
for d in decisions:
    d["equity"] = (equity_vs_random(d["hole"], d["board"], d["n_opponents"], EQ_SAMPLES, d["sequence"])
                   if d in scorable else None)
    d["pot_odds"] = d["to_call"] / (d["pot"] + d["to_call"]) if d["to_call"] else None

showdown_eq = []
for r in results:
    if len(r["showdown"]) >= 2:
        holes = {s: v["cards"] for s, v in r["showdown"].items()}
        for street, k in zip(STREETS, (0, 3, 4, 5)):
            for s, e in equity_known(holes, r["board"][:k], 3000, r["hand_number"] * 10 + k).items():
                showdown_eq.append(dict(hand=r["hand_number"], street=street, seat=s, equity=e))
print(f"own-information equity for {len(scorable)} of {len(decisions)} decisions ({EQ_SAMPLES} samples each); "
      f"{len({e['hand'] for e in showdown_eq})} multi-way showdowns scored by street")

### 0.6 Timing, token and cost metadata from the seat transcripts

For each seat transcript, this cell reads only:

* the timestamps of wakes and of model requests and responses;
* the token-usage and cost fields;
* the tool name and outcome of each tool call.

Each environment tool call is paired, in order, with its receipt. Some calls never reached the environment, so they have no receipt and are counted separately. These are calls the harness stopped (`stale_view`: the state changed after the seat last looked), unknown tools and host errors.

In [ ]:
seat_receipts = {p: [r for _, r in sorted(receipts.items(), key=lambda x: x[1]["sequence"])
                     if r["request"]["method"] in ("invoke", "handle_input") and r["request"].get("participant") == p]
                 for p in players}
attempt_by_seq = {a["sequence"]: a for a in attempts}
hand_end_ms = [commit_ms.get(settled_at.get(h)) for h in range(1, N_HANDS + 1)]

llm_calls, tool_calls, wakes, retries = [], [], [], Counter()
telemetry_seats, seat_meta_rt = [], {}
for p in players:
    recs = transcript_records[p]
    if not recs:
        continue
    telemetry_seats.append(p)
    results_by_call = {r["message"]["toolCallId"]: r["message"] for r in recs
                       if (r.get("message") or {}).get("role") == "toolResult"}
    queue, qi, wake = seat_receipts[p], 0, None
    for rec in recs:
        if rec.get("type") == "model_change":
            seat_meta_rt.setdefault(p, {})["provider/model"] = f"{rec.get('provider')}/{rec.get('modelId')}"
        if rec.get("type") == "service_tier_change":
            seat_meta_rt.setdefault(p, {})["service tier"] = rec.get("serviceTier")
        if rec.get("customType") == "provider_retry_outcome":
            retries[p] += 1
        if rec.get("customType") == "environment_wake":
            wake = {"seat": p, "t": ts_ms(rec["timestamp"]), "first_action_ms": None}
            wakes.append(wake)
            continue
        msg = rec.get("message") or {}
        if msg.get("role") != "assistant":
            continue
        usage = msg.get("usage") or {}
        start, end = msg["timestamp"], ts_ms(rec["timestamp"])
        calls = [c for c in msg.get("content", []) if c.get("type") == "toolCall"]
        llm_calls.append(dict(seat=p, start_ms=start, end_ms=end, latency_s=(end - start) / 1000,
                              input=usage.get("input", 0), output=usage.get("output", 0),
                              cache_read=usage.get("cacheRead", 0), cache_write=usage.get("cacheWrite", 0),
                              total_tokens=usage.get("totalTokens", 0),
                              cost=(usage.get("cost") or {}).get("total", 0), stop=msg.get("stopReason"),
                              n_tools=len(calls)))
        for c in calls:
            name = c["name"]
            res = results_by_call.get(c["id"])
            text = res["content"][0].get("text", "") if res and res.get("content") else ""
            if not text:
                # The turn was aborted before the call returned, so it never reached the environment
                # and has no receipt; only the request is left in the transcript.
                tool_calls.append(dict(seat=p, tool=name, action="say" if name == "say" else name.removeprefix("env_"),
                                       latency_s=(end - start) / 1000, end_ms=end, sequence=None, hand=None,
                                       street=None, outcome="harness: no result (turn aborted)"))
                continue
            try:
                parsed = json.loads(text)
            except ValueError:
                parsed = None
            harness_rejection = (parsed.get("result") or {}).get("rejected") \
                if isinstance(parsed, dict) and isinstance(parsed.get("result"), dict) else None
            action = "say" if name == "say" else name.removeprefix("env_")
            row = dict(seat=p, tool=name, action=action, latency_s=(end - start) / 1000, end_ms=end,
                       sequence=None, hand=None, street=None)
            reaches_env = name == "say" or (name.startswith("env_") and action in ACTIONS)
            host_error = res is not None and res.get("isError") and "environment rejected operation" not in text
            if not reaches_env or host_error:
                row["outcome"] = "harness: unknown tool" if not reaches_env else "harness: host error"
            elif harness_rejection:
                st = (parsed or {}).get("state") or {}
                row.update(outcome=f"harness: {harness_rejection}", hand=st.get("hand_number"), street=st.get("phase"))
            else:
                check(qi < len(queue), f"{p}: tool call {name} has no remaining receipt")
                r = queue[qi]
                qi += 1
                req = r["request"]
                if name == "say":
                    check(req["method"] == "handle_input", f"{p}: say not matched at seq {r['sequence']}")
                else:
                    check(req["method"] == "invoke" and req["params"]["action"] == action
                          and req["params"]["arguments"] == (c.get("arguments") or {}),
                          f"{p}: tool call {name} does not match receipt {r['sequence']}")
                a_ = attempt_by_seq.get(r["sequence"])
                row.update(outcome="accepted" if r["success"] else "rejected", sequence=r["sequence"],
                           hand=a_["hand"] if a_ else None, street=a_["street"] if a_ else None)
                if wake is not None and wake["first_action_ms"] is None:
                    wake["first_action_ms"] = end
            tool_calls.append(row)
    check(qi == len(queue), f"{p}: {len(queue) - qi} receipts were not matched to a transcript tool call")

calls_df, tools_df = pd.DataFrame(llm_calls), pd.DataFrame(tool_calls)
if len(calls_df):
    calls_df["prompt"] = calls_df["input"] + calls_df["cache_read"] + calls_df["cache_write"]
if len(calls_df) and all(hand_end_ms):
    calls_df["hand"] = [min(bisect_left(hand_end_ms, t) + 1, N_HANDS) for t in calls_df["end_ms"]]
print(f"transcripts read for {telemetry_seats}: {len(llm_calls)} model requests, {len(tool_calls)} tool calls, "
      f"{sum(t['sequence'] is not None for t in tool_calls)} matched to receipts, "
      f"{sum(str(t['outcome']).startswith('harness') for t in tool_calls)} stopped by the harness")

### 0.7 Derived tables

These tables come from the replay and the engine's settled results, and the sections below share them.

In [ ]:
hands = []
for r in results:
    h = r["hand_number"]
    m = hand_meta[h]
    folded_at = {e["seat"]: e["phase"] for e in r["history"] if e["action"] == "fold"}
    winners = sorted({s for pot in r["pots"] for s in pot["winners"]}, key=players.index)
    hands.append(dict(hand=h, button=m["button"], sb=m["sb"], bb=m["bb"], dealt=m["dealt"], board=r["board"],
                      pot=sum(pot["amount"] for pot in r["pots"]), showdown=bool(r["showdown"]),
                      winners=winners, folded_at=folded_at, chip_changes=r["chip_changes"], stacks=r["stacks"],
                      revealed=r["showdown"], refunds=r["refunds"], n_pots=len(r["pots"]),
                      street_reached=STREETS[min(3, max(0, len(r["board"]) - 2))] if r["board"] else "preflop"))

stack_curve = {p: [cfg["starting_chips"]] + [h["stacks"][p] for h in hands] for p in players}
final_stacks = {p: final["players"][p]["stack"] for p in players}
ranking = sorted(players, key=lambda p: (-final_stacks[p], players.index(p)))
rank_of = {p: 1 + sum(v > final_stacks[p] for v in final_stacks.values()) for p in players}

style = []
for p in players:
    mine = [d for d in decisions if d["seat"] == p]
    pre = [d for d in mine if d["street"] == "preflop"]
    post = [d for d in mine if d["street"] != "preflop"]
    dealt = [h for h in hands if p in h["dealt"]]
    saw_flop = [h for h in dealt if len(h["board"]) >= 3 and h["folded_at"].get(p) != "preflop"]
    sd = [h for h in dealt if p in h["revealed"]]
    facing = [d for d in mine if d["to_call"] > 0]
    agg_post = sum(d["move"] == "bet/raise" for d in post)
    calls_post = sum(d["move"] == "call" for d in post)
    style.append({
        "Seat": p, "Hands dealt": len(dealt), "Decisions": len(mine),
        "VPIP %": pct(len({d["hand"] for d in pre if d["move"] in ("call", "bet/raise")}), len(dealt)),
        "PFR %": pct(len({d["hand"] for d in pre if d["move"] == "bet/raise"}), len(dealt)),
        "Aggression (postflop bets+raises / calls)": (agg_post / calls_post) if calls_post else (math.inf if agg_post else None),
        "Fold to a bet %": pct(sum(d["move"] == "fold" for d in facing), len(facing)),
        "Saw flop %": pct(len(saw_flop), len(dealt)),
        "Went to showdown % (of flops seen)": pct(len(sd), len(saw_flop)),
        "Won at showdown %": pct(sum(p in h["winners"] for h in sd), len(sd)),
        "Pots won": sum(p in h["winners"] for h in hands),
        "Pots won without showdown": sum(p in h["winners"] and not h["showdown"] for h in hands),
    })
style_df = pd.DataFrame(style)

# Cross-check the replayed move counts against the engine's own per-seat counters.
for p in players:
    mine = [d for d in decisions if d["seat"] == p]
    eng = final["stats"][p]
    check(eng["folds"] == sum(d["move"] == "fold" for d in mine) and eng["checks"] == sum(d["move"] == "check" for d in mine)
          and eng["calls"] == sum(d["move"] == "call" for d in mine) and eng["raises"] == sum(d["move"] == "bet/raise" for d in mine),
          f"{p}: replayed move counts differ from the engine's stats")
check(all(sum(h["chip_changes"][p] for h in hands) == final_stacks[p] - cfg["starting_chips"] for p in players),
      "per-hand chip changes do not add up to final stacks")
print(f"{N_HANDS} hands, {sum(h['showdown'] for h in hands)} showdowns, {len(decisions)} decisions; "
      "move counts match the engine's counters; chip changes reconcile to final stacks")

## 1. Run settings and seats

Every value below comes from the saved run files: `instance.json`, `run.json`, the checkpoint config, the seat transcripts' model records and the table log.

In [ ]:
start_ms = run_status.get("started_at") or instance.get("created_at")
end_ms = max(commit_ms.values()) if commit_ms else None
policy = instance.get("policy", {})
limits = (instance.get("roster") or {}).get("limits", {})
settings = [
    ("Run name", instance.get("name") or "Texas Hold'em"),
    ("Instance", instance.get("id", RUN_DIR.name)),
    ("Definition", f"{instance.get('definition_id')} (version {str(instance.get('version', ''))[:12]}…)"),
    ("Status", f"{run_status.get('phase', snapshot['phase'])}"),
    ("Started", utc(start_ms) if start_ms else "not recorded"),
    ("Last environment event", utc(end_ms) if end_ms else "not recorded"),
    ("Wall-clock duration", f"{(end_ms - start_ms) / 60000:.1f} min" if start_ms and end_ms else "not recorded"),
    ("Hands", f"{N_HANDS} played of {cfg['hands']} configured"
              + ("" if N_HANDS == cfg["hands"] else " (ended early: fewer than two funded seats)")),
    ("Starting chips", cfg["starting_chips"]),
    ("Blinds", f"{cfg['small_blind']} / {cfg['big_blind']} (fixed); starting stack = {cfg['starting_chips'] / BB:g} big blinds"),
    ("Execution policy", ", ".join(f"{k}={v}" for k, v in policy.items())),
    ("Seat limits", ", ".join(f"{k}={v}" for k, v in limits.items())),
    ("Environment operations", ", ".join(f"{n} {m}" for m, n in n_receipts.items())),
]
show(pd.DataFrame(settings, columns=["Setting", "Value"]))

seat_rows = []
for i, p in enumerate(players):
    s = seat_meta.get(p, {})
    seat_rows.append({
        "Seat": p, "Roster order": i + 1, "Occupant": s.get("occupant"), "Model (roster)": s.get("model"),
        "Thinking": s.get("thinking", "–"),
        "Provider/model (transcript)": seat_meta_rt.get(p, {}).get("provider/model", "–"),
        "Service tier": seat_meta_rt.get(p, {}).get("service tier", "–"),
        "Wakes (run.json)": (run_status.get("wakes") or {}).get(p, "–"),
        "Hands on the button": sum(h["button"] == p for h in hands),
    })
show(pd.DataFrame(seat_rows))

## 2. Final result

Seats are ranked by final stack, and equal stacks tie. *Net* is the final stack minus the starting stack. *bb/hand* is net per hand dealt, in big blinds.

In [ ]:
summary = pd.DataFrame([{
    "Rank": rank_of[p], "Seat": p, "Model": seat_meta.get(p, {}).get("model", "–"),
    "Final stack": final_stacks[p], "Net": final_stacks[p] - cfg["starting_chips"],
    "bb / hand": (final_stacks[p] - cfg["starting_chips"]) / BB / max(1, sum(p in h["dealt"] for h in hands)),
    "Pots won": sum(p in h["winners"] for h in hands),
    "Biggest win": max(h["chip_changes"][p] for h in hands), "Biggest loss": min(h["chip_changes"][p] for h in hands),
    "Busted": "yes" if final_stacks[p] == 0 else "no",
} for p in ranking])
show(summary, signed=("Net", "bb / hand", "Biggest win", "Biggest loss"))
display(Markdown(
    f"**{N_HANDS} hands.** " + "; ".join(f"#{rank_of[p]} {p} **{final_stacks[p]}** ({snum(final_stacks[p] - cfg['starting_chips'], 0)})"
                                          for p in ranking) +
    f". {sum(h['showdown'] for h in hands)} hands reached showdown; {sum(not h['showdown'] for h in hands)} were won uncontested."))

fig, ax = plt.subplots(figsize=(10, 3.8))
xs_h = list(range(0, N_HANDS + 1))
for p in players:
    ax.plot(xs_h, stack_curve[p], color=SEAT_COLOR[p], linestyle=SEAT_STYLE[p], marker="o" if N_HANDS <= 30 else None,
            markersize=4, label=p)
ax.axhline(cfg["starting_chips"], color=NEUTRAL, linewidth=0.8)
ax.set_xlim(0, N_HANDS + max(1, N_HANDS * 0.18))
y_top = max(max(v) for v in stack_curve.values()) * 1.2
ax.set_ylim(0, y_top)
# End labels: seats with equal final stacks share one label; the rest are spaced so they never overlap.
groups = defaultdict(list)
for p in players:
    groups[final_stacks[p]].append(p)
placed = []
for value in sorted(groups):
    y = max(value, placed[-1] + 0.05 * y_top) if placed else value
    placed.append(y)
    names = groups[value] if len(groups[value]) <= 3 else groups[value][:2] + [f"+{len(groups[value]) - 2}"]
    ax.annotate(f"{', '.join(names)} {value}", (xs_h[-1], value), xytext=(xs_h[-1] + 0.15, y), textcoords="data",
                va="center", fontsize=8.5, color=INK2)
ax.xaxis.set_major_locator(MaxNLocator(integer=True))
ax.set_title("Chip stacks after each hand (gray line = starting stack)")
ax.set_xlabel("Hand")
ax.set_ylabel("Chips")
ax.legend(loc="upper left", ncol=min(len(players), 5), fontsize=8.5)
finish(fig)

grid = [[h["chip_changes"][p] for h in hands] for p in players]
lim = max(1, max(abs(v) for row in grid for v in row))
fig, ax = plt.subplots(figsize=(10, 0.9 + 0.42 * len(players)))
im = ax.imshow(grid, aspect="auto", cmap=DIVERGING, norm=TwoSlopeNorm(0, -lim, lim))
ax.set_yticks(range(len(players)), players)
ax.set_xticks(range(N_HANDS), [str(h["hand"]) if N_HANDS <= 30 or h["hand"] % 5 == 0 else "" for h in hands])
if N_HANDS <= 30:
    for i, row in enumerate(grid):
        for j, v in enumerate(row):
            if v:
                ax.text(j, i, snum(v, 0), ha="center", va="center", fontsize=8, color="#0b0b0b")
ax.grid(False)
ax.set_xlabel("Hand")
ax.set_title("Chip change by hand (blue = won, red = lost)")
fig.colorbar(im, ax=ax, pad=0.01, fraction=0.03)
finish(fig)

### Highlights

These sentences are generated from §4–§10 and describe this run only. They are descriptive, not causal.

In [ ]:
winner = ranking[0]
lines = [f"**{winner}** finished first with {final_stacks[winner]} chips "
         f"({snum(final_stacks[winner] - cfg['starting_chips'], 0)}), winning {sum(winner in h['winners'] for h in hands)} pots."]
big = max(hands, key=lambda h: h["pot"])
lines.append(f"The biggest pot was hand {big['hand']} ({big['pot']} chips, {big['street_reached']}"
             f"{', showdown' if big['showdown'] else ', uncontested'}), won by {', '.join(big['winners'])}.")
for s in style:
    lines.append(f"{s['Seat']}: VPIP {num(s['VPIP %'], 0)}%, PFR {num(s['PFR %'], 0)}%, "
                 f"folded to {num(s['Fold to a bet %'], 0)}% of bets faced.")
folds_strong = [d for d in decisions if d["move"] == "fold" and d["equity"] is not None and d["pot_odds"] is not None
                and d["equity"] > d["pot_odds"]]
calls_weak = [d for d in decisions if d["move"] == "call" and d["equity"] is not None and d["pot_odds"] is not None
              and d["equity"] < d["pot_odds"]]
lines.append(f"Against random hands, {len(folds_strong)} folds had more raw equity than the price, and "
             f"{len(calls_weak)} calls had less (see §6; real ranges are stronger than random).")
if len(calls_df):
    act_ = tools_df[tools_df["outcome"].isin(["accepted", "rejected"])]
    lines.append("Median model response time per action: " +
                 ", ".join(f"{p} {act_[act_['seat'] == p]['latency_s'].median():.1f} s" for p in telemetry_seats) + ".")
    lines.append(f"Recorded model cost: **${calls_df['cost'].sum():.4f}** (" +
                 ", ".join(f"{p} ${calls_df[calls_df['seat'] == p]['cost'].sum():.4f}" for p in telemetry_seats) +
                 f"), {int(calls_df['total_tokens'].sum()):,} tokens.")
display(Markdown("\n".join(f"- {line}" for line in lines)))

## 3. Hand by hand

Each row is one settled hand (*Actions*: P = preflop, F = flop, T = turn, R = river). Showdown cards were public at the table. The other hole cards (in faded text) were private while the hand was played and come from each seat's own view after the run finished. `??` marks a holding that was never recorded. *Actions* lists every decision as *street seat action*.

In [ ]:
abbrev = {"preflop": "P", "flop": "F", "turn": "T", "river": "R"}
rows = []
for h in hands:
    m = hand_meta[h["hand"]]
    acts = [d for d in decisions if d["hand"] == h["hand"]]
    holes = []
    for p in m["dealt"]:
        cell = cards_html(m["holes"][p], hidden=p in m["hidden"])
        if p in h["revealed"]:
            cell += f" <b>{esc(h['revealed'][p]['hand'])}</b>"
        elif p not in m["hidden"]:
            cell = f"<span class='private'>{cell}</span>"
        if p in h["folded_at"]:
            cell += f" <i>fold {abbrev[h['folded_at'][p]]}</i>"
        holes.append(f"<div class='cell-left'>{esc(p)}: {cell}</div>")
    rows.append({
        "Hand": h["hand"], "Button / SB / BB": esc(f"{h['button']} / {h['sb']} / {h['bb']}"),
        "Hole cards": "".join(holes), "Board": f"<div class='cell-left'>{cards_html(h['board'])}</div>", "Pot": h["pot"],
        "Ended": (f"showdown, {h['n_pots']} pots" if h["showdown"] and h["n_pots"] > 1 else
                  "showdown" if h["showdown"] else f"uncontested ({h['street_reached']})"),
        "Winner(s)": esc(", ".join(h["winners"])),
        "Actions": "<div class='cell-wrap'>" + esc("; ".join(f"{abbrev[d['street']]} {d['seat']} {d['action']}"
                                 + (f" {d['amount']}" if d["amount"] is not None else "") for d in acts)) + "</div>",
        **{f"{p} Δ": h["chip_changes"][p] for p in players},
    })
show(pd.DataFrame(rows), escape=False, signed=tuple(f"{p} Δ" for p in players))

## 4. Where the chips came from

The tables below split each seat's net chips three ways: by table position, by how the hand ended (showdown or uncontested), and across the biggest pots. Each split adds up to the seat's net result.

In [ ]:
positions = ["Button", "Button (SB)", "Small blind", "Big blind", "Other"]
pos_rows = []
for p in players:
    row = {"Seat": p}
    for pos in positions:
        hs = [h for h in hands if p in h["dealt"] and position_of(p, h["hand"]) == pos]
        if any(position_of(q, h["hand"]) == pos for h in hands for q in hand_meta[h["hand"]]["dealt"]):
            row[f"{pos} (hands)"] = len(hs)
            row[f"{pos} net"] = sum(h["chip_changes"][p] for h in hs)
    row["Showdown net"] = sum(h["chip_changes"][p] for h in hands if h["showdown"])
    row["Uncontested net"] = sum(h["chip_changes"][p] for h in hands if not h["showdown"])
    row["Total net"] = final_stacks[p] - cfg["starting_chips"]
    check(row["Showdown net"] + row["Uncontested net"] == row["Total net"], "position split does not reconcile")
    pos_rows.append(row)
pos_df = pd.DataFrame(pos_rows)
show(pos_df, places=0, signed=tuple(c for c in pos_df.columns if c.endswith("net")))

used_pos = [pos for pos in positions if f"{pos} net" in pos_df.columns]
fig, axes = plt.subplots(1, 2, figsize=(12, 3.6), gridspec_kw={"width_ratios": [len(used_pos), 2]})
width = 0.8 / len(players)
for ax, cats, key in ((axes[0], used_pos, lambda c: f"{c} net"), (axes[1], ["Showdown", "Uncontested"], lambda c: f"{c} net")):
    for i, p in enumerate(players):
        vals = [pos_df.loc[pos_df["Seat"] == p, key(c)].iloc[0] for c in cats]
        bars = ax.bar([j + (i - (len(players) - 1) / 2) * width for j in range(len(cats))], vals, width=width * 0.9,
                      color=SEAT_COLOR[p], label=p)
        if len(players) <= 4:
            ax.bar_label(bars, labels=[snum(v, 0) for v in vals], fontsize=7.5, color=INK2, padding=2)
    ax.axhline(0, color=NEUTRAL, linewidth=0.8)
    ax.set_xticks(range(len(cats)), cats)
    ax.margins(y=0.18)
axes[0].set_title("Net chips by position")
axes[1].set_title("Net chips by how the hand ended")
axes[0].legend(fontsize=8, ncol=min(len(players), 4))
finish(fig)

top = sorted(hands, key=lambda h: -h["pot"])[:min(5, N_HANDS)]
show(pd.DataFrame([{"Hand": h["hand"], "Pot": h["pot"], "Street reached": h["street_reached"],
                    "Ended": "showdown" if h["showdown"] else "uncontested", "Board": cards_html(h["board"]),
                    "Winner(s)": esc(", ".join(h["winners"])),
                    "Winning hand": esc(", ".join(h["revealed"][w]["hand"] for w in h["winners"] if w in h["revealed"]) or "–"),
                    **{f"{p} Δ": h["chip_changes"][p] for p in players}} for h in top]),
     escape=False, signed=tuple(f"{p} Δ" for p in players))

## 5. Playing style

These are standard poker tendency statistics, computed from the replayed decisions.

* **VPIP**: the share of hands dealt in which the seat put chips in voluntarily preflop. Posting a blind does not count.
* **PFR**: the share of hands dealt in which the seat raised preflop.
* **Aggression**: postflop bets and raises divided by postflop calls. ∞ means the seat raised but never called.
* **Fold to a bet**: folds as a share of the decisions where the seat owed chips.
* **Showdown rates**: how often a seat that saw the flop reached showdown, and how often it won there.

*n/a* means the seat never reached the situation the statistic measures. With a handful of hands these are descriptions of this run, not stable player profiles.

In [ ]:
show(style_df, places=0)

metrics = ["VPIP %", "PFR %", "Fold to a bet %", "Saw flop %", "Went to showdown % (of flops seen)", "Won at showdown %"]
fig, axes = plt.subplots(1, len(metrics), figsize=(13, 0.9 + 0.38 * len(players)), sharey=True)
for ax, mname in zip(axes, metrics):
    raw = [style_df.loc[style_df["Seat"] == p, mname].iloc[0] for p in players]
    defined = [v is not None and not (isinstance(v, float) and math.isnan(v)) for v in raw]
    vals = [v if ok else 0 for v, ok in zip(raw, defined)]
    bars = ax.barh(range(len(players)), vals, color=[SEAT_COLOR[p] for p in players], height=0.7)
    ax.bar_label(bars, labels=[f"{v:.0f}" if ok else "n/a" for v, ok in zip(vals, defined)], fontsize=8, color=INK2, padding=2)
    ax.set_xlim(0, 115)
    ax.set_title(mname.replace(" (of flops seen)", "").replace(" %", " %"), fontsize=9)
    ax.set_xticks([0, 50, 100])
axes[0].set_yticks(range(len(players)), players)
axes[0].invert_yaxis()
finish(fig)

mix = []
for p in players:
    for st_ in STREETS:
        ds = [d for d in decisions if d["seat"] == p and d["street"] == st_]
        mix.append({"Seat": p, "Street": st_, "Decisions": len(ds), **{mv: sum(d["move"] == mv for d in ds) for mv in MOVES}})
show(pd.DataFrame(mix), places=0, collapsed="<b>Moves by seat and street</b> (click to expand)")

fig, axes = plt.subplots(1, 4, figsize=(13, 3.3), sharey=True)
width = 0.8 / len(players)
for ax, st_ in zip(axes, STREETS):
    for i, p in enumerate(players):
        row = next(m for m in mix if m["Seat"] == p and m["Street"] == st_)
        ax.bar([j + (i - (len(players) - 1) / 2) * width for j in range(len(MOVES))], [row[mv] for mv in MOVES],
               width=width * 0.9, color=SEAT_COLOR[p], label=p)
    ax.set_xticks(range(len(MOVES)), MOVES, fontsize=8)
    ax.set_title(st_.capitalize())
axes[0].set_ylabel("Decisions")
axes[0].yaxis.set_major_locator(MaxNLocator(integer=True))
axes[0].legend(fontsize=8)
finish(fig)

## 6. Decision quality: own-information equity against the price

Each accepted decision gets the actor's **own-information equity** (see §0.5): its estimated share of the pot against random holdings for every opponent still in the hand. When the seat owed chips, the **price** (pot odds) is `to call / (pot + to call)`. Calling is break-even on immediate odds when equity equals price.

* A **fold above the line** gave up a hand with more raw equity than the price.
* A **call below the line** paid more than the raw equity justified.

Opponents' real ranges are stronger than random after they bet or raise, so this baseline flatters calls and makes folds look worse than they were. Read it as a consistency check, not a verdict. Implied odds and future betting are ignored.

In [ ]:
eq_rows = []
for p in players:
    ds = [d for d in decisions if d["seat"] == p and d["equity"] is not None]
    facing = [d for d in ds if d["pot_odds"] is not None]
    eq_rows.append({
        "Seat": p, "Scored decisions": len(ds),
        **{f"Mean equity: {mv}": (statistics.mean(d["equity"] for d in ds if d["move"] == mv)
                                  if any(d["move"] == mv for d in ds) else None) for mv in MOVES},
        "Facing a bet": len(facing),
        "Calls/raises with equity ≥ price": sum(d["move"] != "fold" and d["equity"] >= d["pot_odds"] for d in facing),
        "Calls below price": sum(d["move"] == "call" and d["equity"] < d["pot_odds"] for d in facing),
        "Folds above price": sum(d["move"] == "fold" and d["equity"] > d["pot_odds"] for d in facing),
        "Folds below price": sum(d["move"] == "fold" and d["equity"] <= d["pot_odds"] for d in facing),
    })
show(pd.DataFrame(eq_rows), places=2)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.4))
jit = random.Random(0)
for i, p in enumerate(players):
    ds = [d for d in decisions if d["seat"] == p and d["equity"] is not None]
    offset = (i - (len(players) - 1) / 2) * (0.6 / len(players))
    axes[0].scatter([MOVES.index(d["move"]) + offset + jit.uniform(-0.04, 0.04) for d in ds], [d["equity"] for d in ds],
                    s=34, color=SEAT_COLOR[p], edgecolor="white", linewidth=1, label=p)
axes[0].set_xticks(range(len(MOVES)), MOVES)
axes[0].set_ylim(-0.02, 1.02)
axes[0].set_ylabel("Own-information equity")
axes[0].set_title("Equity behind each move")
axes[0].legend(fontsize=8)

MARK = {"fold": "x", "call": "o", "bet/raise": "^", "check": "s"}
for p in players:
    for mv in ("fold", "call", "bet/raise"):
        ds = [d for d in decisions if d["seat"] == p and d["equity"] is not None and d["pot_odds"] is not None and d["move"] == mv]
        if ds:
            axes[1].scatter([d["pot_odds"] for d in ds], [d["equity"] for d in ds], marker=MARK[mv], s=44,
                            color=SEAT_COLOR[p], linewidths=1.6 if mv == "fold" else 1,
                            edgecolor=None if mv == "fold" else "white", label=f"{p} {mv}")
axes[1].plot([0, 1], [0, 1], color=NEUTRAL, linewidth=1)
axes[1].set_xlim(0, max(0.6, max([d["pot_odds"] for d in decisions if d["pot_odds"] is not None] or [0]) + 0.05))
axes[1].set_ylim(-0.02, 1.02)
axes[1].set_xlabel("Price: to call / (pot + to call)")
axes[1].set_ylabel("Own-information equity")
axes[1].set_title("Facing a bet: equity vs price (gray = break-even)")
axes[1].legend(fontsize=7, ncol=2, loc="upper right")
finish(fig)

flagged = [d for d in decisions if d["equity"] is not None and d["pot_odds"] is not None and
           ((d["move"] == "fold" and d["equity"] > d["pot_odds"]) or (d["move"] == "call" and d["equity"] < d["pot_odds"]))]
if flagged:
    show(pd.DataFrame([{"Seq": d["sequence"], "Hand": d["hand"], "Street": d["street"], "Seat": d["seat"], "Move": d["move"],
                        "Hole": cards_html(d["hole"]), "Board": cards_html(d["board"]), "Opponents": d["n_opponents"],
                        "Pot": d["pot"], "To call": d["to_call"], "Price": d["pot_odds"], "Equity": d["equity"],
                        "Hand result Δ": hands[d["hand"] - 1]["chip_changes"][d["seat"]]} for d in flagged]),
         escape=False, signed=("Hand result Δ",), collapsed=f"<b>{len(flagged)} decisions off the break-even line</b> (click to expand)")

## 7. Showdowns: equity by street against the revealed hands

For every hand that reached showdown, this section computes each player's share of the pot at each street. The share is taken against the hands that were actually revealed, with the remaining board cards still to come. This is hindsight, not something the players could know. It shows who was ahead when the money went in and how much the runout changed. Players who folded before showdown are not included.

In [ ]:
if not showdown_eq:
    display(Markdown("_No hand reached a showdown in this run._"))
else:
    sd_df = pd.DataFrame(showdown_eq)
    rows = []
    for h in sorted(sd_df["hand"].unique()):
        hh = hands[h - 1]
        for s in hh["revealed"]:
            e = {st_: sd_df[(sd_df["hand"] == h) & (sd_df["seat"] == s) & (sd_df["street"] == st_)]["equity"].iloc[0] for st_ in STREETS}
            rows.append({"Hand": h, "Pot": hh["pot"], "Seat": s, "Hole": cards_html(hh["revealed"][s]["cards"]),
                         "Board": cards_html(hh["board"]), **{f"{st_.capitalize()} equity %": 100 * e[st_] for st_ in STREETS},
                         "Final hand": esc(hh["revealed"][s]["hand"]), "Chips Δ": hh["chip_changes"][s]})
    show(pd.DataFrame(rows), places=0, escape=False, signed=("Chips Δ",))

    shown = sorted(sd_df["hand"].unique(), key=lambda h: -hands[h - 1]["pot"])[:12]
    shown = sorted(shown)
    cols = min(4, len(shown))
    nrows = math.ceil(len(shown) / cols)
    fig, axes = plt.subplots(nrows, cols, figsize=(3.2 * cols, 2.6 * nrows), squeeze=False, sharey=True)
    for ax, h in zip(axes.flat, shown):
        for s in hands[h - 1]["revealed"]:
            ys = [100 * sd_df[(sd_df["hand"] == h) & (sd_df["seat"] == s) & (sd_df["street"] == st_)]["equity"].iloc[0] for st_ in STREETS]
            ax.plot(range(4), ys, color=SEAT_COLOR[s], linestyle=SEAT_STYLE[s], marker="o", markersize=4, label=s)
        ax.set_xticks(range(4), ["pre", "flop", "turn", "river"], fontsize=8)
        ax.set_ylim(-3, 103)
        ax.set_title(f"Hand {h} · pot {hands[h - 1]['pot']}", fontsize=9)
        ax.legend(fontsize=7)
    for ax in list(axes.flat)[len(shown):]:
        ax.axis("off")
    axes[0][0].set_ylabel("Pot share %")
    if len(sd_df["hand"].unique()) > len(shown):
        fig.suptitle(f"Largest {len(shown)} showdown pots", fontsize=9, x=0.01, ha="left")
    finish(fig)

## 8. Time to decide

Timing comes from the seat transcripts.

* **Model response time** runs from sending a model request to receiving the response that carries the action. If a request fails (for example a provider rate limit) and is retried, only the final request's time counts here. **Wake to action** below includes the retries.
* **Wake to first action** runs from the environment's wake message to the first environment action the seat sent in response.
* **Idle-retry wakes** are extra wakes from the harness after a seat's turn ended without an action.

The harness's own `time_to_action` counters from `run.json` are shown for comparison. All times are wall-clock and include provider latency.

In [ ]:
if not len(tools_df):
    display(Markdown("_No seat transcripts available: timing cannot be reported for this run._"))
else:
    act = tools_df[tools_df["outcome"].isin(["accepted", "rejected"])]
    tta = run_status.get("time_to_action") or {}
    trows = []
    for p in telemetry_seats:
        a_ = act[act["seat"] == p]["latency_s"]
        w_ = [(w["first_action_ms"] - w["t"]) / 1000 for w in wakes if w["seat"] == p and w["first_action_ms"]]
        h_ = tta.get(p) or {}
        trows.append({"Seat": p, "Action calls": len(a_), "Median response (s)": a_.median() if len(a_) else None,
                      "Mean (s)": a_.mean() if len(a_) else None, "Max (s)": a_.max() if len(a_) else None,
                      "Wakes": sum(w["seat"] == p for w in wakes), "Idle-retry wakes": status_wakes[p],
                      "Wakes with an action": len(w_),
                      "Median wake→action (s)": statistics.median(w_) if w_ else None,
                      "Harness time_to_action mean (s)": h_["total_ms"] / h_["count"] / 1000 if h_.get("count") else None,
                      "Harness max (s)": h_.get("max_ms", 0) / 1000 if h_ else None})
    show(pd.DataFrame(trows))

    order = [a for a in ACTIONS if a in set(act["action"])]
    fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))
    nseat = len(telemetry_seats)
    for i, p in enumerate(telemetry_seats):
        data = [act[(act["seat"] == p) & (act["action"] == a)]["latency_s"].tolist() for a in order]
        pos = [j + (i - (nseat - 1) / 2) * (0.8 / nseat) for j in range(len(order))]
        keep = [(d_, x) for d_, x in zip(data, pos) if d_]
        if keep:
            bp = axes[0].boxplot([d_ for d_, _ in keep], positions=[x for _, x in keep], widths=0.7 / nseat,
                                 patch_artist=True, medianprops={"color": "white", "linewidth": 1.5},
                                 flierprops={"marker": "o", "markersize": 3, "markerfacecolor": SEAT_COLOR[p], "markeredgecolor": "none"},
                                 whiskerprops={"color": SEAT_COLOR[p]}, capprops={"color": SEAT_COLOR[p]})
            for box in bp["boxes"]:
                box.set(facecolor=SEAT_COLOR[p], edgecolor=SEAT_COLOR[p])
        axes[0].plot([], [], color=SEAT_COLOR[p], linewidth=6, label=p)
        med = act[(act["seat"] == p) & act["street"].notna()].groupby("street")["latency_s"].median().reindex(STREETS)
        axes[1].plot(range(4), med.values, color=SEAT_COLOR[p], linestyle=SEAT_STYLE[p], marker="o", label=p)
        s_ = act[(act["seat"] == p) & act["hand"].notna()].sort_values("end_ms")
        axes[2].scatter(s_["hand"], s_["latency_s"], s=12, color=SEAT_COLOR[p], alpha=0.45, edgecolor="none")
        roll = s_.groupby("hand")["latency_s"].median()
        axes[2].plot(roll.index, roll.values, color=SEAT_COLOR[p], linestyle=SEAT_STYLE[p], label=f"{p} (median per hand)")
    axes[0].set_xticks(range(len(order)), [o.replace("_", "\n") for o in order])
    axes[0].set_ylabel("Model response time (s)")
    axes[0].set_title("Decision time by action")
    axes[0].legend(fontsize=8)
    axes[1].set_xticks(range(4), STREETS)
    axes[1].set_title("Median decision time by street")
    axes[1].set_ylim(bottom=0)
    axes[1].legend(fontsize=8)
    axes[2].set_xlabel("Hand")
    axes[2].set_title("Decision time over the run")
    axes[2].xaxis.set_major_locator(MaxNLocator(integer=True))
    axes[2].set_ylim(bottom=0)
    axes[2].legend(fontsize=8)
    finish(fig)

## 9. Cost and token usage

Usage and cost are the per-request fields that the harness recorded in each seat transcript. With prompt caching, a request's prompt is split across *input*, *cache read* and *cache write*. **Prompt** is the sum of the three. Cost is computed from the model catalog's prices at run time, so the provider's actual bill may differ. **Failed requests** ended in an error, such as a provider rate limit, and the harness retried them. Each request is assigned to a hand by its completion time: a hand ends when the operation that settled it commits.

In [ ]:
if not len(calls_df):
    display(Markdown("_No seat transcripts available: usage cannot be reported for this run._"))
else:
    crows = []
    for p in telemetry_seats + ["Total"]:
        d = calls_df if p == "Total" else calls_df[calls_df["seat"] == p]
        n_dec = len(decisions) if p == "Total" else sum(x["seat"] == p for x in decisions)
        crows.append({"Seat": p, "Model requests": len(d), "Requests with tool calls": int((d["n_tools"] > 0).sum()),
                      "Failed requests": int((d["stop"] == "error").sum()),
                      "Retry outcomes": sum(retries.values()) if p == "Total" else retries[p],
                      "Input tokens": int(d["input"].sum()), "Cache read": int(d["cache_read"].sum()),
                      "Cache write": int(d["cache_write"].sum()), "Mean prompt / request": d[d["stop"] != "error"]["prompt"].mean(),
                      "Output tokens": int(d["output"].sum()), "Total tokens": int(d["total_tokens"].sum()),
                      "Cost (USD)": f"${d['cost'].sum():.4f}", "Cost / hand": f"${d['cost'].sum() / N_HANDS:.4f}",
                      "Cost / decision": f"${d['cost'].sum() / n_dec:.4f}" if n_dec else "–"})
    show(pd.DataFrame(crows), places=0)
    display(Markdown(f"**Total recorded model cost: ${calls_df['cost'].sum():.4f}** "
                     f"({int(calls_df['total_tokens'].sum()):,} tokens across {len(calls_df)} requests)."))

    fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
    for p in telemetry_seats:
        d = calls_df[calls_df["seat"] == p].sort_values("end_ms")
        if "hand" in d:
            per_hand = d.groupby("hand")["cost"].sum().reindex(range(1, N_HANDS + 1), fill_value=0)
            axes[0].plot([0] + list(per_hand.index), [0] + list(per_hand.cumsum().values), color=SEAT_COLOR[p],
                         linestyle=SEAT_STYLE[p], label=p)
            tok = d.groupby("hand")["total_tokens"].sum().reindex(range(1, N_HANDS + 1), fill_value=0)
            axes[1].plot(tok.index, tok.values / 1000, color=SEAT_COLOR[p], linestyle=SEAT_STYLE[p], marker="o", markersize=3, label=p)
        ok = d[d["stop"] != "error"]
        axes[2].scatter(range(1, len(ok) + 1), ok["prompt"] / 1000, s=10, color=SEAT_COLOR[p], alpha=0.7, edgecolor="none", label=p)
    axes[0].set_title("Cumulative cost (USD)")
    axes[0].set_xlabel("Hand")
    axes[1].set_title("Tokens per hand (thousands)")
    axes[1].xaxis.set_major_locator(MaxNLocator(integer=True))
    axes[1].set_xlabel("Hand")
    axes[1].set_ylim(bottom=0)
    axes[2].set_title("Prompt tokens per successful request (thousands)")
    axes[2].set_xlabel("Request # (per seat)")
    axes[2].set_ylim(bottom=0)
    for ax in axes:
        ax.legend(fontsize=8)
    finish(fig)

## 10. Action mix and rejections

The **environment** counts every attempt that reached it, accepted or rejected, and each one has a receipt. The **harness** may stop a tool call before it reaches the environment: a stale view, an unknown tool or a host error. Those calls have no receipt and are counted separately.

In [ ]:
mix_rows = []
for p in players:
    mine = [a for a in attempts if a["seat"] == p]
    row = {"Seat": p, **{a: sum(x["success"] and x["action"] == a for x in mine) for a in ACTIONS}}
    row["Rejected by environment"] = sum(not x["success"] for x in mine)
    row["Stopped by harness"] = int(((tools_df["seat"] == p) & tools_df["outcome"].astype(str).str.startswith("harness")).sum()) if len(tools_df) else 0
    mix_rows.append(row)
mix_df = pd.DataFrame(mix_rows)
show(mix_df, places=0)

cats = [c for c in mix_df.columns if c != "Seat"]
fig, ax = plt.subplots(figsize=(10, 0.6 + 0.5 * len(cats)))
hgt = 0.8 / len(players)
for i, p in enumerate(players):
    vals = [mix_df.loc[mix_df["Seat"] == p, c].iloc[0] for c in cats]
    bars = ax.barh([j + (i - (len(players) - 1) / 2) * hgt for j in range(len(cats))], vals, height=hgt * 0.9,
                   color=SEAT_COLOR[p], label=p)
    ax.bar_label(bars, fontsize=7.5, color=INK2, padding=2)
ax.set_yticks(range(len(cats)), cats)
ax.invert_yaxis()
ax.set_title("Tool calls by type and outcome")
ax.legend(fontsize=8)
finish(fig)

rej = Counter((a["seat"], a["action"], str(a["result"])) for a in attempts if not a["success"])
if len(tools_df):
    for (s, outcome), n in tools_df[tools_df["outcome"].astype(str).str.startswith("harness")].groupby(["seat", "outcome"]).size().items():
        rej[(s, "(any)", outcome)] += n
if rej:
    show(pd.DataFrame([{"Seat": s, "Attempted action": a_, "Error / harness outcome": err, "Count": n}
                       for (s, a_, err), n in sorted(rej.items())]))

## 11. Complete action log

Every attempt is listed in receipt order (accepted and rejected) and grouped by hand. Expand a hand to see its rows. *Before* is the acting seat's situation when it acted: its cards, the board, the pot, the amount owed and its stack. *Equity* is the own-information equity from §0.5. States are rebuilt by the replay; they are not copied from tool responses.

In [ ]:
head = ("<tr><th>Seq</th><th>Street</th><th>Seat</th><th>Position</th><th>Action</th><th>OK</th><th>Hole</th><th>Board</th>"
        "<th>Pot</th><th>To call</th><th>Stack</th><th>Equity</th><th>Price</th><th>Legal</th><th>Result / error</th></tr>")
blocks = []
for h in hands:
    rows_html = []
    for a in [x for x in attempts if x["hand"] == h["hand"]]:
        legal = ", ".join(k if not isinstance(v, dict) else f"{k} {v['minimum']}–{v['maximum']}"
                          for k, v in a["legal"].items() if v) or "none"
        args = f" {a['amount']}" if a["amount"] is not None else ""
        res = (f"{a['move']}, paid {a['paid']}" if a["success"] else a["result"])
        rows_html.append(
            f"<tr><td>{a['sequence']}</td><td>{a['street']}</td><td>{esc(a['seat'])}</td><td>{esc(a['position'])}</td>"
            f"<td>{esc(a['action'] + args)}</td><td>{'✓' if a['success'] else '✗'}</td>"
            f"<td>{cards_html(a['hole'], hidden=not a['hole_known'])}</td><td>{cards_html(a['board'])}</td>"
            f"<td>{a['pot']}</td><td>{a['to_call']}</td><td>{a['stack']}</td>"
            f"<td>{num(a.get('equity'))}</td><td>{num(a.get('pot_odds'))}</td>"
            f"<td style='text-align:left'>{esc(legal)}</td><td style='text-align:left'>{esc(res)}</td></tr>")
    n = sum(x["hand"] == h["hand"] for x in attempts)
    tail = f"pot {h['pot']}, won by {', '.join(h['winners'])}" + (" at showdown" if h["showdown"] else "")
    blocks.append(f"<details{' open' if h['hand'] == 1 else ''}><summary><b>Hand {h['hand']}</b>: {n} attempts · {esc(tail)}</summary>"
                  f"<table class='dataframe' border='0'><thead>{head}</thead><tbody>{''.join(rows_html)}</tbody></table></details>")
display(Markdown(f"**{len(attempts)} attempts: {len(decisions)} accepted, {len(attempts) - len(decisions)} rejected.**"))
display(HTML("".join(blocks)))

## 12. Verification results

Every check below has already passed as an assertion in the cells above; this cell would not run otherwise. The table records each check and its values.

In [ ]:
n_hidden = sum(len(m["hidden"]) for m in hand_meta.values())
checks = [
    ("Engine", f"Replay used the run's own engine and evaluator from definition version {str(instance['version'])[:12]}…", True),
    ("Receipt replay", f"All {sum(n_receipts.values())} receipts replayed in order; every result, notice and progress record matches", True),
    ("Rejections", f"All {len(attempts) - len(decisions)} rejected attempts raised the recorded error and left state unchanged", True),
    ("Chip conservation", f"Stacks + chips in the pot = {CHIPS_TOTAL} after every operation", True),
    ("Final state", "Rebuilt state equals checkpoint.snapshot (stacks, results, stats, history, last hand's cards); "
                    "only the never-recorded undealt deck is excluded", True),
    ("Engine counters", "Replayed fold/check/call/raise counts equal the engine's per-seat stats", True),
    ("Chip changes", "Per-hand chip changes add up to each seat's final stack", True),
    ("Wake states", f"{wakecount} transcript wake states verified against the replayed agent view"
                    + (f"; {len(late_wakes)} late queued wake(s) were handed over while the seat had already acted, "
                       f"so they carry the view of the operation committed at delivery (up to {max(delivered - op for _, _, op, delivered in late_wakes)} "
                       "operations past their own)" if late_wakes else ""),
     not wake_problems),
    ("Hole-card sources", f"No conflicts across own views, showdowns and the final snapshot; {n_hidden} seat-hand(s) never recorded", not hole_conflicts),
    ("Transcript alignment", f"{sum(t['sequence'] is not None for t in tool_calls)} receipts matched to transcript tool calls "
                             f"(action + arguments); {sum(str(t['outcome']).startswith('harness') for t in tool_calls)} harness-stopped calls had no receipt",
     bool(tool_calls) or not telemetry_seats),
    ("Final ranking", " / ".join(f"#{rank_of[p]} {p} {final_stacks[p]}" for p in ranking), True),
    ("Privacy", "Run finished before any private card is shown; no assistant text or reasoning is displayed", True),
]
show(pd.DataFrame([{"Check": c, "Detail": d, "Result": "PASS" if ok else "FAIL"} for c, d, ok in checks]))
assert all(ok for _, _, ok in checks)

## 13. Methods and limitations

**Source of truth.** All game facts come from replaying the run's own `checkpoint.json` receipts through the environment's engine (`environment.py` and `poker_eval.py`) from the definition version the run used. The replay deals each hand's recorded board and recovered hole cards. Spare cards stand in for burns and for holdings that were never recorded; these never affect betting, pots or settlement, because a holding that was never recorded never reached showdown. The rebuilt final state must equal the saved snapshot. No game is started, resumed or modified, and nothing is written to disk.

**Private cards.** Folded hands were never public at the table. This report shows them after the run has finished, taken from each seat's own view of its cards, and marks them as private. A seat's reasoning is never shown.

**Own-information equity.** This is a Monte Carlo estimate of pot share against uniformly random holdings for every live opponent, with the remaining board drawn at random. It ignores what the opponents' actions reveal, so it understates opponents' strength after they bet. It also ignores future betting, implied odds and fold equity. Use it to compare decisions, not to grade them.

**Showdown equity.** This is exact enumeration on the flop and turn, and a 3,000-sample estimate preflop. It uses only the hands revealed at showdown, so it is hindsight and leaves out players who folded earlier.

**Timing and cost.** Times come from transcript timestamps and include network and provider latency. Cost is the per-request figure that the harness recorded from its model price catalog, not an invoice.

**Limitations.**

* A short run is a small sample, so the style and decision statistics describe this run only.
* Equity figures are estimates with sampling noise of about ±1–2 percentage points at the default sample size.
* A wake queued while its seat was still acting is handed over only later, with the fresh view: the state of
  the operation that had committed at that moment, which can be several operations past its own cursor. Such wakes
  are verified against that operation and marked late.
* Calls that the harness stopped before they reached the environment are known only from the transcripts.